In [2]:
#Importing dependencies 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

In [3]:
df = pd.read_csv("/users/imbahndu/Desktop/Columbia DBM/Typing dataset/modeling.csv")

In [4]:
df

,Hand,HoldTime,Direction,LatencyTime,FlightTime,Parkinsons,UPDRS,Gender_Male,Ratio,Latency-HoldTime
0,0,101.6,0,234.4,156.3,True,0,False,0.650032,132.8
1,0,78.1,0,210.9,125.0,True,0,False,0.624800,132.8
2,1,85.9,1,195.3,117.2,True,0,False,0.732935,109.4
3,1,78.1,1,203.1,171.9,True,0,False,0.454334,125.0
4,0,78.1,0,203.1,132.8,True,0,False,0.588102,125.0
...,...,...,...,...,...,...,...,...,...,...
1123031,1,109.4,3,296.9,187.5,True,0,False,0.583467,187.5
1123032,1,132.8,3,320.3,156.3,True,0,False,0.849648,187.5
1123033,1,117.2,3,281.3,210.9,True,0,False,0.555714,164.1
1123034,1,109.4,3,203.1,93.8,True,0,False,1.166311,93.7


In [5]:
df.isnull().sum()

Hand                0
HoldTime            0
Direction           0
LatencyTime         0
FlightTime          0
Parkinsons          0
UPDRS               0
Gender_Male         0
Ratio               0
Latency-HoldTime    0
dtype: int64

In [6]:
df["Parkinsons"].value_counts()


Parkinsons
True     740327
False    382709
Name: count, dtype: int64

In [7]:
df["Gender_Male"].value_counts()

Gender_Male
False    635192
True     487844
Name: count, dtype: int64

In [8]:
#Dataset was already preprocessed and balanced, so jump right into modeling 

#defining labels
from sklearn.model_selection import train_test_split

y = df["Parkinsons"]
X = df.drop(columns = ["Parkinsons", "Hand", "Direction", "UPDRS"])

#Balancing dataset
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state = 42)

X_train_raw, y_train_raw = smote.fit_resample(X, y)

X_train, X_test, y_train, y_test = train_test_split(X_train_raw, y_train_raw, test_size = 0.2, random_state = 42)

/users/imbahndu/.local/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


In [9]:
print(X_train)

print(y_train)

           HoldTime  LatencyTime  FlightTime  Gender_Male     Ratio  \
611423   109.400000   281.300000  140.600000         True  0.778094   
89586     80.100000   193.400000  144.500000        False  0.554325   
404661    85.900000   296.900000  218.800000         True  0.392596   
1479579   91.800000   211.091011  119.100000         True  0.770781   
865292    89.800000   238.000000  125.000000        False  0.718400   
...             ...          ...         ...          ...       ...   
259178    80.100000   271.500000  183.600000         True  0.436275   
1414414  120.330148   184.422207  103.884926         True  1.158315   
131932   121.100000   300.800000  207.000000         True  0.585024   
671155    78.100000   234.400000  136.700000        False  0.571324   
121958   125.000000   226.600000  125.000000        False  1.000000   

         Latency-HoldTime  
611423         171.900000  
89586          113.300000  
404661         211.000000  
1479579        119.291011  
865292 

In [10]:
#training model
model_xgb = XGBClassifier()

model_xgb.fit(X_train, y_train)

pred = model_xgb.predict(X_test)

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

acc_score = accuracy_score(y_test, pred)

acc_score

0.7445319807787769

In [11]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

       False       0.75      0.74      0.74    148308
        True       0.74      0.75      0.75    147823

    accuracy                           0.74    296131
   macro avg       0.74      0.74      0.74    296131
weighted avg       0.74      0.74      0.74    296131



In [12]:
#Class is so freaking imbalanced--> will adjust this demain using sklearn.utils resample 

#This is the best i could get so keeping this and try sklearn.utils demain and also maybe try SMOTE and RandomOverSampler on the entire dataset. Do this for both tyoing and this dataset

In [13]:
feat = model_xgb.feature_importances_

In [14]:
pd.DataFrame({"Column": X_train.columns, "Feature" :feat}).sort_values(by = "Feature", ascending = False)

,Column,Feature
3,Gender_Male,0.751616
0,HoldTime,0.099733
2,FlightTime,0.047526
5,Latency-HoldTime,0.040078
1,LatencyTime,0.039052
4,Ratio,0.021994


In [15]:
#Training a Random Forest Classifer
from sklearn.ensemble import RandomForestClassifier

random = RandomForestClassifier()

random.fit(X_train, y_train)

pred = random.predict(X_test)

print(accuracy_score(y_test, pred))

0.7615717368326855


In [16]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

       False       0.76      0.76      0.76    148308
        True       0.76      0.76      0.76    147823

    accuracy                           0.76    296131
   macro avg       0.76      0.76      0.76    296131
weighted avg       0.76      0.76      0.76    296131



In [17]:
print(confusion_matrix(y_test, pred))

[[112920  35388]
 [ 35218 112605]]


In [18]:
#Keep random forest
ft_rf = random.feature_importances_

In [19]:
#Feature selection 
pd.DataFrame({"Column": X_train.columns, "Feature" :ft_rf}).sort_values(by = "Feature", ascending = False)

,Column,Feature
3,Gender_Male,0.423032
4,Ratio,0.120017
1,LatencyTime,0.116463
0,HoldTime,0.115084
5,Latency-HoldTime,0.112818
2,FlightTime,0.112587


In [20]:
#Retrain this model and, extract features for deployment and add to /.dart page

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

       False       0.76      0.76      0.76    148308
        True       0.76      0.76      0.76    147823

    accuracy                           0.76    296131
   macro avg       0.76      0.76      0.76    296131
weighted avg       0.76      0.76      0.76    296131



In [23]:
#Saving this model, because GridSearchCV take a lot of computing time so yeah 

import joblib 
joblib.dump(random, "typing_page.pkl")

['typing_page.pkl']